# Vaccine Sense - Aplicação 19

## Treinamento do modelo

A Aplicação 18 terminou com o arquivo `vaccinesense_dataset.csv`, já rotulado.
Aqui vamos transformar esse CSV em um modelo `modelo_vaccinesense.pkl`.

```
CSV rotulado -> treino -> teste -> modelo .pkl
```


## 1. Pacotes


In [ ]:
!pip -q install pandas matplotlib scikit-learn==1.6.1 joblib


## 2. Imports


In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


## 3. Abrir o CSV do app18

Selecione o arquivo `vaccinesense_dataset.csv` que você baixou no final da Aplicação 18.


In [ ]:
arquivos = files.upload()
ARQUIVO_CSV = next(iter(arquivos))

df = pd.read_csv(ARQUIVO_CSV)
print(f"{len(df)} medições carregadas")
df.head()


## 4. Definir a pergunta do modelo

O dataset possui três situações, mas o modelo responderá a uma pergunta binária:
**a carga está em perigo?**

- `TRANSPORTE_OK` e `AMBIENTE_HOSTIL` viram `0`;
- `CARGA_EM_PERIGO` vira `1`.

O ambiente hostil permanece no treino para ensinar que temperatura externa alta, sozinha, não significa que a carga está em perigo.


In [ ]:
df["situacao"] = df["situacao"].str.strip().str.upper()
df["alvo"] = (df["situacao"] == "CARGA_EM_PERIGO").astype(int)

display(df[["situacao", "alvo"]].value_counts().sort_index())


## 5. Remover leituras fora do protocolo

Algumas leituras foram registradas enquanto os sliders ainda estavam mudando. Vamos remover somente temperaturas internas fora da faixa planejada para cada rodada. O CSV original não é alterado.


In [ ]:
LIMITES_TEMP_INTERNA = {
    1: (3.0, 6.0), 2: (6.8, 7.9), 3: (4.2, 5.8),
    4: (3.0, 6.0), 5: (6.8, 7.9), 6: (4.2, 5.8),
    7: (8.3, 10.0), 8: (4.0, 6.0), 9: (4.0, 6.0),
    10: (4.0, 7.0), 11: (4.0, 7.0), 12: (2.2, 3.5),
    13: (0.5, 1.8), 14: (6.3, 7.8), 15: (2.5, 3.8),
    16: (4.0, 7.0), 17: (4.0, 6.0), 18: (4.2, 5.8),
}
TOLERANCIA = 0.1

limite_inferior = df["rodada"].map(
    lambda rodada: LIMITES_TEMP_INTERNA[int(rodada)][0]
)
limite_superior = df["rodada"].map(
    lambda rodada: LIMITES_TEMP_INTERNA[int(rodada)][1]
)

fora_do_protocolo = (
    (df["tempInterna"] < limite_inferior - TOLERANCIA)
    | (df["tempInterna"] > limite_superior + TOLERANCIA)
)

print(f"{fora_do_protocolo.sum()} leituras removidas")
display(df.loc[fora_do_protocolo, ["rodada", "id", "tempInterna"]])

df = df.loc[~fora_do_protocolo].copy()
print(f"{len(df)} medições prontas para o treinamento")


## 6. Separar as cinco features

Entram somente as medições úteis. A umidade permaneceu constante nesta coleta, então fica fora do modelo. `timestamp`, `device`, `rodada`, `id` e `tempoForaDaFaixa` também não são features.


In [ ]:
FEATURES = [
    "tempInterna",
    "tempExterna",
    "luz",
    "criticidade",
    "distancia",
]

X = df[FEATURES]
y = df["alvo"]

display(X.head())


## 7. Separar 70% para treino e 30% para teste

Primeiro embaralhamos as medições. A estratificação por `rodada` garante que todas as 18 condições apareçam no treino e no teste.

Esta divisão avalia novas medições das condições conhecidas. Para avaliar uma viagem completamente nova, seria necessário repetir cada cenário em outra rodada.


In [ ]:
indices_treino, indices_teste = train_test_split(
    df.index,
    test_size=0.30,
    random_state=42,
    stratify=df["rodada"],
)

X_treino = X.loc[indices_treino]
X_teste = X.loc[indices_teste]
y_treino = y.loc[indices_treino]
y_teste = y.loc[indices_teste]

print(f"Treino: {len(X_treino)} medições")
print(f"Teste:  {len(X_teste)} medições")
print(f"Rodadas no treino: {df.loc[indices_treino, 'rodada'].nunique()}")
print(f"Rodadas no teste:  {df.loc[indices_teste, 'rodada'].nunique()}")


## 8. Procurar o melhor modelo

O `GridSearchCV` treina diferentes modelos e diferentes hiperparâmetros. A melhor combinação será a que obtiver o maior F1 para `CARGA_EM_PERIGO`.

A validação usa cinco divisões embaralhadas dos dados de treino.


In [ ]:
pipeline = Pipeline([
    ("escala", "passthrough"),
    ("classificador", LogisticRegression()),
])

param_grid = [
    {
        "escala": [StandardScaler()],
        "classificador": [LogisticRegression(max_iter=1000, random_state=42)],
        "classificador__C": [0.1, 1, 10],
    },
    {
        "escala": ["passthrough"],
        "classificador": [DecisionTreeClassifier(random_state=42)],
        "classificador__max_depth": [3, 5, None],
        "classificador__min_samples_leaf": [1, 5],
    },
    {
        "escala": ["passthrough"],
        "classificador": [RandomForestClassifier(random_state=42)],
        "classificador__n_estimators": [50, 100],
        "classificador__max_depth": [5, None],
        "classificador__min_samples_leaf": [1, 2],
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

busca = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

busca.fit(X_treino, y_treino)
modelo = busca.best_estimator_
melhor_nome = type(modelo.named_steps["classificador"]).__name__

print(f"Melhor modelo: {melhor_nome}")
print(f"Melhor F1 na validação: {busca.best_score_:.3f}")
print("Melhores parâmetros:")
for nome, valor in busca.best_params_.items():
    print(f"  {nome}: {valor}")

resultados = pd.DataFrame(busca.cv_results_)
display(
    resultados[["rank_test_score", "mean_test_score", "params"]]
    .sort_values("rank_test_score")
    .head(5)
)


## 9. Testar e interpretar


In [ ]:
predicoes = modelo.predict(X_teste)

print(f"Acurácia: {accuracy_score(y_teste, predicoes) * 100:.1f}%")
print()
print(classification_report(
    y_teste,
    predicoes,
    target_names=["TRANSPORTE_OK", "CARGA_EM_PERIGO"],
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_teste,
    predicoes,
    display_labels=["TRANSPORTE_OK", "CARGA_EM_PERIGO"],
    cmap="Blues",
)
plt.title(f"Matriz de confusão - {melhor_nome}")
plt.show()

importancias = permutation_importance(
    modelo,
    X_teste,
    y_teste,
    scoring="f1",
    n_repeats=10,
    random_state=42,
)

importancia_df = (
    pd.DataFrame({
        "feature": FEATURES,
        "importancia": importancias.importances_mean,
    })
    .sort_values("importancia")
)

display(importancia_df.sort_values("importancia", ascending=False))
importancia_df.plot.barh(x="feature", y="importancia", legend=False)
plt.title("Importância das features no conjunto de teste")
plt.xlabel("Queda média do F1 ao embaralhar a feature")
plt.show()


## 10. Fazer um `.predict()`

É exatamente esta chamada que a aplicação Python fará com a última medição recebida do InfluxDB.


In [ ]:
uma_medicao = X_teste.iloc[[0]]
predicao = modelo.predict(uma_medicao)[0]
resultado = "CARGA_EM_PERIGO" if predicao == 1 else "TRANSPORTE_OK"

display(uma_medicao)
print(f"Resultado: {resultado}")


## 11. Treinar com todos os dados e salvar o modelo

A acurácia acima continua sendo a do teste 70/30. Agora repetimos o treinamento do melhor modelo com todas as medições antes de gerar o arquivo final. Baixe o `.pkl` e coloque-o na pasta `app` da Aplicação 19.


In [ ]:
MODELO_ARQUIVO = "modelo_vaccinesense.pkl"

modelo_final = clone(modelo)
modelo_final.fit(X, y)

joblib.dump(modelo_final, MODELO_ARQUIVO)
print(f"Modelo salvo em {MODELO_ARQUIVO}")
files.download(MODELO_ARQUIVO)


---

## Próximo passo

A aplicação Python consulta a última linha do InfluxDB, monta estas mesmas cinco colunas e executa:

```python
predicao = modelo.predict(medicao)[0]
```
